# Assignment 3: Network Biology

**Group:** Group 02  
**Members:** Claire Bams, Yustyna Babichuk, Antonia Constantin, Francisco Javier Camacho Perez de Sevilla

This notebook compares the normal Boolean regulatory network with four mutated versions. Each network will be tested with the same scenarios, attractor analysis, and basin-size analysis.

## 1. Setup

In [31]:
import numpy as np
import pandas as pd
from itertools import product
from collections import defaultdict
import matplotlib.pyplot as plt

## 2. Boolean network model

In [32]:
# This class stores the node states and Boolean update rules.
class BooleanNetwork:
    def __init__(self, node_names):
        # Every node starts OFF (0). A scenario will set the starting values later.
        self.nodes = {name: 0 for name in node_names}
        self.rules = {}
        self.history = []

    def add_rule(self, target_node, rule_function, rule_description=""):
        # Save both the executable rule and a readable description of it.
        self.rules[target_node] = {
            'function': rule_function,
            'description': rule_description
        }

    def set_state(self, **kwargs):
        # Set the requested node values and convert True/False to 1/0.
        for node, value in kwargs.items():
            if node in self.nodes:
                self.nodes[node] = int(bool(value))

    def get_state_vector(self):
        # Always use alphabetical node order so results can be compared safely.
        return [self.nodes[node] for node in sorted(self.nodes.keys())]

    def update_synchronous(self):
        # Calculate every new value from the same current state.
        new_state = {}
        for node in self.nodes:
            if node in self.rules:
                new_state[node] = int(self.rules[node]['function'](self.nodes))
            else:
                new_state[node] = self.nodes[node]

        self.nodes = new_state
        self.history.append(self.get_state_vector())

    def simulate(self, steps=10, record_history=True, verbose=False):
        # Store the starting state, then update until stable or out of steps.
        if record_history:
            self.history = [self.get_state_vector()]

        for step in range(steps):
            self.update_synchronous()

            # Two identical consecutive states mean a fixed point was reached.
            if len(self.history) >= 2 and self.history[-1] == self.history[-2]:
                if verbose:
                    print(f"Reached steady state after {step + 1} steps")
                break

        return np.array(self.history)

In [33]:
# Build a new, independent copy of the normal eight-node network.
# Always call this function before applying a mutation.
def create_normal_network():
    nodes = [
        'DNA_damage', 'p53', 'MYC', 'CDK2',
        'MDM2', 'p21', 'Growth', 'Death'
    ]

    network = BooleanNetwork(nodes)

    # Normal Boolean rules copied from the completed practical.
    network.add_rule('DNA_damage', lambda s: s['DNA_damage'], "DNA_damage = INPUT (constant)")
    network.add_rule('p21', lambda s: s['p53'], "p21 = p53")
    network.add_rule('MYC', lambda s: (not s['p53']) and (not s['p21']), "MYC = (NOT p53) AND (NOT p21)")
    network.add_rule('CDK2', lambda s: s['MYC'] and (not s['p21']) and (not s['p53']), "CDK2 = MYC AND (NOT p21) AND (NOT p53)")
    network.add_rule('MDM2', lambda s: s['MYC'], "MDM2 = MYC")
    network.add_rule('p53', lambda s: s['DNA_damage'] and (not s['MDM2']), "p53 = DNA_damage AND (NOT MDM2)")
    network.add_rule('Growth', lambda s: s['CDK2'] and s['MYC'] and (not s['p53']), "Growth = CDK2 AND MYC AND (NOT p53)")
    network.add_rule('Death', lambda s: s['p53'] and s['DNA_damage'] and (not s['Growth']), "Death = p53 AND DNA_damage AND (NOT Growth)")

    return network

## 3. Shared analysis functions

In [34]:
# Starting states required by the practical and reused for every network.
SCENARIOS = {
    'Healthy Cell': {
        'DNA_damage': 0, 'p53': 0, 'MYC': 0, 'CDK2': 0,
        'MDM2': 0, 'p21': 0, 'Growth': 0, 'Death': 0
    },
    'Stressed Cell': {
        'DNA_damage': 1, 'p53': 0, 'MYC': 0, 'CDK2': 0,
        'MDM2': 0, 'p21': 0, 'Growth': 0, 'Death': 0
    },
    'Oncogene Hijacked Cell': {
        'DNA_damage': 0, 'p53': 0, 'MYC': 1, 'CDK2': 0,
        'MDM2': 0, 'p21': 0, 'Growth': 0, 'Death': 0
    }
}


# Run all three scenarios and collect the required final outputs.
def run_scenarios(network, steps=15):
    rows = []
    trajectories = {}
    node_names = sorted(network.nodes.keys())

    for scenario_name, initial_state in SCENARIOS.items():
        # Reset the network to this scenario before starting the simulation.
        network.set_state(**initial_state)
        trajectory = network.simulate(steps=steps)
        trajectories[scenario_name] = trajectory

        # Convert the final state vector back into named node values.
        final_state = {
            node: int(trajectory[-1][i])
            for i, node in enumerate(node_names)
        }

        # The assignment specifically asks for Growth, Death, and p53.
        rows.append({
            'Scenario': scenario_name,
            'Growth': final_state['Growth'],
            'Death': final_state['Death'],
            'p53': final_state['p53']
        })

    return pd.DataFrame(rows), trajectories

In [35]:
# Test all 2^8 starting states and collect unique fixed-point attractors.
def find_attractors(network, max_steps=15):
    attractors = []
    node_names = sorted(network.nodes.keys())

    # product([0, 1], repeat=8) generates all 256 possible starting states.
    for initial_state in product([0, 1], repeat=len(node_names)):
        network.set_state(**dict(zip(node_names, initial_state)))
        trajectory = network.simulate(steps=max_steps)

        # A fixed point has identical states in the final two time steps.
        reached_fixed_point = (
            len(trajectory) >= 2
            and np.array_equal(trajectory[-1], trajectory[-2])
        )

        if reached_fixed_point:
            final_state = tuple(int(x) for x in trajectory[-1])
            if final_state not in attractors:
                attractors.append(final_state)

    return attractors


# Give each attractor a consistent biological interpretation.
def classify_attractor(attractor, node_names):
    state = dict(zip(node_names, attractor))

    if state['Growth'] == 1 and state['Death'] == 0 and state['DNA_damage'] == 0:
        return 'Healthy growth'
    if state['Death'] == 1 and state['Growth'] == 0:
        return 'Cell death'
    if state['Growth'] == 1 and state['Death'] == 0 and state['DNA_damage'] == 1:
        return 'Cancer-like growth'
    return 'Other/conflicting state'

In [36]:
# Count how many of the 256 initial states lead to each attractor.
def calculate_basins(network, attractors, max_steps=15):
    node_names = sorted(network.nodes.keys())
    basin_data = defaultdict(list)

    # This lookup connects each final state to its attractor number.
    attractor_lookup = {
        tuple(int(x) for x in attractor): index
        for index, attractor in enumerate(attractors)
    }

    all_states = list(product([0, 1], repeat=len(node_names)))

    for initial_state in all_states:
        network.set_state(**dict(zip(node_names, initial_state)))
        trajectory = network.simulate(steps=max_steps)

        reached_fixed_point = (
            len(trajectory) >= 2
            and np.array_equal(trajectory[-1], trajectory[-2])
        )

        if reached_fixed_point:
            final_state = tuple(int(x) for x in trajectory[-1])
            if final_state in attractor_lookup:
                attractor_index = attractor_lookup[final_state]
                basin_data[attractor_index].append(initial_state)

    # Make one readable row for every attractor.
    rows = []
    for index, attractor in enumerate(attractors):
        basin_size = len(basin_data[index])
        rows.append({
            'Attractor': index + 1,
            'State': tuple(int(x) for x in attractor),
            'Classification': classify_attractor(attractor, node_names),
            'Basin size': basin_size,
            'Basin percentage': 100 * basin_size / len(all_states)
        })

    return pd.DataFrame(rows)


# Run the complete required analysis for one normal or mutated network.
def analyse_network(network):
    scenario_results, trajectories = run_scenarios(network)
    attractors = find_attractors(network)
    basin_results = calculate_basins(network, attractors)

    return {
        'scenarios': scenario_results,
        'trajectories': trajectories,
        'attractors': attractors,
        'basins': basin_results
    }

## 4. Normal-network baseline

In [37]:
# Create and analyse the normal network
normal_network = create_normal_network()
normal_results = analyse_network(normal_network)

# Show the final Growth, Death, and p53 values for all three scenarios
print("NORMAL NETWORK — SCENARIO ANALYSIS")
display(normal_results['scenarios'])

# Show every attractor, its classification, basin size, and percentage
print("\nNORMAL NETWORK — ATTRACTOR AND BASIN ANALYSIS")
display(normal_results['basins'])

# Count the number of attractors
number_of_attractors = len(normal_results['attractors'])
print(f"\nNumber of attractors: {number_of_attractors}")

# Select all attractors classified as cancer-like
cancer_like_attractors = normal_results['basins'][
    normal_results['basins']['Classification'] == 'Cancer-like growth'
]

# Add their basin sizes and percentages
cancer_like_basin_size = cancer_like_attractors['Basin size'].sum()
cancer_like_percentage = cancer_like_attractors['Basin percentage'].sum()

print(f"Cancer-like basin size: {cancer_like_basin_size} out of 256 states")
print(f"Percentage leading to cancer-like growth: {cancer_like_percentage:.1f}%")

NORMAL NETWORK — SCENARIO ANALYSIS


,Scenario,Growth,Death,p53
0,Healthy Cell,1,0,0
1,Stressed Cell,0,1,1
2,Oncogene Hijacked Cell,1,0,0



NORMAL NETWORK — ATTRACTOR AND BASIN ANALYSIS


,Attractor,State,Classification,Basin size,Basin percentage
0,1,"(1, 0, 0, 1, 1, 1, 0, 0)",Healthy growth,128,50.000
1,2,"(0, 1, 1, 0, 0, 0, 1, 1)",Cell death,120,46.875
2,3,"(1, 1, 0, 1, 1, 1, 0, 0)",Cancer-like growth,8,3.125



Number of attractors: 3
Cancer-like basin size: 8 out of 256 states
Percentage leading to cancer-like growth: 3.1%


## 5. Mutation A — p53 knockout

In [38]:
mutation_a = create_normal_network()

# TODO: Remove the # from the next line to apply the required p53 knockout.
mutation_a.add_rule('p53', lambda s: False, "p53 = BROKEN (always OFF)")

# TODO: Remove the # from these lines to run and display the analysis.
mutation_a_results = analyse_network(mutation_a)
mutation_a_results['scenarios']
mutation_a_results['basins']

display(mutation_a_results['scenarios'])
display(mutation_a_results['basins'])

# Percentage calculation
basins = mutation_a_results['basins']
is_cancer_like = basins['Classification'] == 'Cancer-like growth' 
cancer_like_a = basins[is_cancer_like]  

print(f"Cancer-like basin size: {cancer_like_a['Basin size'].sum()} out of 256 states")
print(f"Percentage leading to cancer-like growth: {cancer_like_a['Basin percentage'].sum():.1f}%")

,Scenario,Growth,Death,p53
0,Healthy Cell,1,0,0
1,Stressed Cell,1,0,0
2,Oncogene Hijacked Cell,1,0,0


,Attractor,State,Classification,Basin size,Basin percentage
0,1,"(1, 0, 0, 1, 1, 1, 0, 0)",Healthy growth,128,50.0
1,2,"(1, 1, 0, 1, 1, 1, 0, 0)",Cancer-like growth,128,50.0


Cancer-like basin size: 128 out of 256 states
Percentage leading to cancer-like growth: 50.0%


### Interpretation:
As seen in the table, the percentage of states leadin to cancer-like growth is exactly 50.0% out of all 256.
Inactivating p53 eliminates the ability of the cell to trigger apoptpsis in response to DNA damage. p1 can no longer activate nor can it trigger the Death node. Consequently, the protective "Cell death" attractor  disappears and every state that would normally result in programmed cell death now stabilizes into uncontrolled, cancer-like growth, which increases from 3.1% to 50% of all possible network states.

## 6. Mutation B — MYC amplification

In [39]:
mutation_b = create_normal_network()

# TODO: Remove the # from the next line to apply the required MYC amplification.
mutation_b.add_rule('MYC', lambda s: True, "MYC = AMPLIFIED (always ON)")

# TODO: Remove the # from these lines to run and display the analysis.
mutation_b_results = analyse_network(mutation_b)
mutation_b_results['scenarios']
mutation_b_results['basins']

display(mutation_b_results['scenarios'])
display(mutation_b_results['basins'])

# Percentage calculation
basins = mutation_b_results['basins']
is_cancer_like = basins['Classification'] == 'Cancer-like growth' #
cancer_like_b = basins[is_cancer_like]  

print(f"Cancer-like basin size: {cancer_like_b['Basin size'].sum()} out of 256 states")
print(f"Percentage leading to cancer-like growth: {cancer_like_b['Basin percentage'].sum():.1f}%")

,Scenario,Growth,Death,p53
0,Healthy Cell,1,0,0
1,Stressed Cell,1,0,0
2,Oncogene Hijacked Cell,1,0,0


,Attractor,State,Classification,Basin size,Basin percentage
0,1,"(1, 0, 0, 1, 1, 1, 0, 0)",Healthy growth,128,50.0
1,2,"(1, 1, 0, 1, 1, 1, 0, 0)",Cancer-like growth,128,50.0


Cancer-like basin size: 128 out of 256 states
Percentage leading to cancer-like growth: 50.0%


### Interpretation:
Like with the p53 inactivation, the percentage of states leading to the cancer-like growth attractor is 50.0%.
Amplifying the MYC oncogene forces it to remain constantly active. This permanent MYC activity drives the overexpression of MDM2, which in turn  degrades p53. Because p53 is suppressed entirely by the excess MDM2, the cell is blind to DNA damage and it loses its ability to trigger apoptosis, making 50% of all possible initial states to stabilize into cancer-like growth instead of cell death.

## 7. Mutation C — MDM2 overexpression

In [40]:
mutation_c = create_normal_network()

# TODO: Remove the # from the next line to apply the required MDM2 overexpression.

mutation_c.add_rule('MDM2', lambda s: True, "MDM2 = OVEREXPRESSED (always ON)")


# TODO: Remove the # from these lines to run and display the analysis.
mutation_c_results = analyse_network(mutation_c)
# mutation_c_results['scenarios']
# mutation_c_results['basins']

# Record the scenario outcomes and basin results.
display(mutation_c_results['scenarios'])
display(mutation_c_results['basins'])

# Calculate the percentage of states leading to cancer-like growth.----------------


basins = mutation_c_results['basins']

is_cancer_like = basins['Classification'] == 'Cancer-like growth' #

cancer_like_c = basins[is_cancer_like]  # Keep only the rows marked True




print(f"Cancer-like basin size: {cancer_like_c['Basin size'].sum()} out of 256 states")
print(f"Percentage leading to cancer-like growth: {cancer_like_c['Basin percentage'].sum():.1f}%")

,Scenario,Growth,Death,p53
0,Healthy Cell,1,0,0
1,Stressed Cell,1,0,0
2,Oncogene Hijacked Cell,1,0,0


,Attractor,State,Classification,Basin size,Basin percentage
0,1,"(1, 0, 0, 1, 1, 1, 0, 0)",Healthy growth,128,50.0
1,2,"(1, 1, 0, 1, 1, 1, 0, 0)",Cancer-like growth,128,50.0


Cancer-like basin size: 128 out of 256 states
Percentage leading to cancer-like growth: 50.0%


### Short interpretation:

When MDM2 is always ON, 50.0% of the starting states (128 out of 256) end in cancer-like growth. In the normal network it is only 3.1% ( from section 4). MDM2 switches p53 OFF, even when there is DNA damage. Without p53, the cell does not die and keeps growing. You can see this in the table: the Stressed Cell now has Growth = 1 and Death = 0.



## 8. Mutation D — p21(CDKN1A) knockout (group-selected mutation)

In [41]:
mutation_d = create_normal_network()

# TODO: Add the group-selected mutation rule here.

#  p21 (CDKN1A) knockout
mutation_d.add_rule('p21', lambda s: False, "p21 = KNOCKED OUT (always OFF)")

# TODO: Remove the # from these lines after adding the mutation rule.
mutation_d_results = analyse_network(mutation_d)
# mutation_d_results['scenarios']
# mutation_d_results['basins']

mutation_d_results = analyse_network(mutation_d)
display(mutation_d_results['scenarios'])
display(mutation_d_results['basins'])



basins = mutation_d_results['basins']

is_cancer_like = basins['Classification'] == 'Cancer-like growth' #

cancer_like_d = basins[is_cancer_like]



print(f"Cancer-like basin size: {cancer_like_d['Basin size'].sum()} out of 256 states")
print(f"Percentage leading to cancer-like growth: {cancer_like_d['Basin percentage'].sum():.1f}%")

,Scenario,Growth,Death,p53
0,Healthy Cell,1,0,0
1,Stressed Cell,0,1,0
2,Oncogene Hijacked Cell,1,0,0


,Attractor,State,Classification,Basin size,Basin percentage
0,1,"(1, 0, 0, 1, 1, 1, 0, 0)",Healthy growth,128,50.000
1,2,"(0, 1, 1, 0, 0, 0, 0, 1)",Cell death,24,9.375
2,3,"(1, 1, 0, 1, 1, 1, 0, 0)",Cancer-like growth,8,3.125


Cancer-like basin size: 8 out of 256 states
Percentage leading to cancer-like growth: 3.1%


In [42]:
mutation_d.set_state(**SCENARIOS['Stressed Cell'])

trajectory = mutation_d.simulate(steps=15)

for i, row in enumerate(trajectory):

    print(i, dict(zip(sorted(mutation_d.nodes.keys()), map(int, row))))

0 {'CDK2': 0, 'DNA_damage': 1, 'Death': 0, 'Growth': 0, 'MDM2': 0, 'MYC': 0, 'p21': 0, 'p53': 0}
1 {'CDK2': 0, 'DNA_damage': 1, 'Death': 0, 'Growth': 0, 'MDM2': 0, 'MYC': 1, 'p21': 0, 'p53': 1}
2 {'CDK2': 0, 'DNA_damage': 1, 'Death': 1, 'Growth': 0, 'MDM2': 1, 'MYC': 0, 'p21': 0, 'p53': 1}
3 {'CDK2': 0, 'DNA_damage': 1, 'Death': 1, 'Growth': 0, 'MDM2': 0, 'MYC': 0, 'p21': 0, 'p53': 0}
4 {'CDK2': 0, 'DNA_damage': 1, 'Death': 0, 'Growth': 0, 'MDM2': 0, 'MYC': 1, 'p21': 0, 'p53': 1}
5 {'CDK2': 0, 'DNA_damage': 1, 'Death': 1, 'Growth': 0, 'MDM2': 1, 'MYC': 0, 'p21': 0, 'p53': 1}
6 {'CDK2': 0, 'DNA_damage': 1, 'Death': 1, 'Growth': 0, 'MDM2': 0, 'MYC': 0, 'p21': 0, 'p53': 0}
7 {'CDK2': 0, 'DNA_damage': 1, 'Death': 0, 'Growth': 0, 'MDM2': 0, 'MYC': 1, 'p21': 0, 'p53': 1}
8 {'CDK2': 0, 'DNA_damage': 1, 'Death': 1, 'Growth': 0, 'MDM2': 1, 'MYC': 0, 'p21': 0, 'p53': 1}
9 {'CDK2': 0, 'DNA_damage': 1, 'Death': 1, 'Growth': 0, 'MDM2': 0, 'MYC': 0, 'p21': 0, 'p53': 0}
10 {'CDK2': 0, 'DNA_damage': 1

### Short interpretation:

When p21 is knocked out, the cell has trouble dying. In the normal network, 46.9% of states led to cell death. Now only 9.4% do. Most states (50.0%) still lead to healthy growth, and cancer-like growth stays low, at 3.1%. So this mutation does not create more cancer directly, but it makes the cell's death response weaker.

## 9. Comparison of all networks

| Network | Cancer-like basin size | Cancer-like percentage |
|---|---:|---:|
| Normal network | 8 | 3.1% |
| p53 knockout | 128 | 50.0% |
| MYC amplification | 128 | 50.0% |
| MDM2 overexpression | 128 | 50.0% |
| p21 knockout | 8 | 3.1% |

The p21 knockout does not increase the cancer-like fixed-point basin, which remains at 3.1%. However, 37.5% of the initial states no longer reach a fixed point and instead show oscillatory behaviour.


### Comparison of cancer-like basin percentages

- The normal network has a cancer-like basin of **3.1%**.
- p53 knockout, MYC amplification, and MDM2 overexpression increase this to **50.0%**, an increase of **46.9 percentage points**.
- The p21 knockout keeps the cancer-like fixed-point basin at **3.1%**, but many states become oscillatory instead of reaching a fixed point.

### Which mutation is the most dangerous?

- The p53 knockout, MYC amplification, and MDM2 overexpression show the strongest effect in this model. For all three mutations, 50.0% of the initial states lead to cancer-like growth, compared with only 3.1% in the normal network.

### What is the role of feedback loops?

- The MYC → MDM2 ─| p53 interaction is important because MYC activates MDM2, while MDM2 inhibits p53. Since p53 normally suppresses growth and promotes cell death, increased MYC or MDM2 activity can keep p53 inactive and favour continued growth.

### What are the limitations of this Boolean model?

- Each node can only be **ON or OFF**, so intermediate levels of gene or protein activity are not represented.
- All nodes are updated at the **same time**, while biological processes can happen at different speeds.
- The network contains only a small number of genes and interactions, so many biological mechanisms and external factors are not included.



## 10. Conclusion

Overall, the mutations had a clear effect on the behaviour of the Boolean network. p53 knockout, MYC amplification, and MDM2 overexpression produced the largest increase in cancer-like states, while p21 knockout mainly introduced more oscillatory behaviour. The analysis also shows how feedback interactions can strongly influence cell fate in a simplified regulatory model.